In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline

# ============================================================
# STEP 0 — Load raw datasets from your folder
# ============================================================

BASE = "data/macro_processed"

paths = {
    # Returns building blocks
    "sp500":        f"{BASE}/other/sp500_processed.csv",
    "tbill_3m":     f"{BASE}/other/3m_yield_processed.csv",

    # Inflation
    "cpi":          f"{BASE}/inflation/cpi_processed.csv",
    "pce":          f"{BASE}/inflation/PCE_price_index_processed.csv",
    "ppi":          f"{BASE}/inflation/PPI_inflation_processed.csv",

    # Growth
    "indprod":      f"{BASE}/ec_growth/industrial_production_processed.csv",
    "retail_sales": f"{BASE}/ec_growth/retail_sales_processed.csv",
    "inventories":  f"{BASE}/ec_growth/tot_business_inventories_processed.csv",
    "export_px":    f"{BASE}/ec_growth/export_price_index_processed.csv",
    "import_px":    f"{BASE}/ec_growth/import_price_index_processed.csv",
    "unemp":        f"{BASE}/ec_growth/unemployment_processed.csv",

    # Money & policy
    "m2":           f"{BASE}/mon_policy/m2_real_money_supply_processed.csv",
    "fedfunds":     f"{BASE}/mon_policy/fedfunds_processed.csv",
    "discount_rate":f"{BASE}/mon_policy/fed_reserve_discount_rate_processed.csv",

    # Yields & spreads
    "spread_10y_2y":f"{BASE}/mkt_vol/10y_2y_spread_processed.csv",
    "nat_fin_cond": f"{BASE}/mkt_vol/nat_fin_condition_indx_processed.csv",
    "nasdaq_vol":   f"{BASE}/mkt_vol/nasdaq_vol_indx_processed.csv",
    "hy_spread":    f"{BASE}/other/bofa_highyield_spread_processed.csv",
    "y2":           f"{BASE}/other/2y_yield_processed.csv",
    "y3m":          f"{BASE}/other/3m_yield_processed.csv",
    "y10":          f"{BASE}/other/10y_yield_processed.csv",
}


# ============================================================
# STEP 1 — BUILD EMRP VIA GEOMETRIC RETURNS
# ============================================================

# --------- S&P 500 geometric monthly return ---------
sp = pd.read_csv(paths["sp500"], parse_dates=["date"]).set_index("date")
sp = sp.sort_index()
sp["sp500_geo"] = sp["pct_change_mom"] / 100.0     # monthly %
# Already monthly, but ensure month-end alignment
sp_m = sp.resample("M").last()[["sp500_geo"]]

# --------- 3M T-bill geometric monthly return ---------
rf = pd.read_csv(paths["tbill_3m"], parse_dates=["date"]).set_index("date")
rf = rf.sort_index()

# daily annualized % yield → daily simple return
rf["r_daily"] = (rf["value"] / 100) / 252

# geometric compounding inside each month
rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
rf_m = rf_m.to_frame(name="rf_geo")

# --------- EMRP ---------
df = sp_m.join(rf_m, how="inner")
df["EMRP_geo"] = df["sp500_geo"] - df["rf_geo"]
df["EMRP_next"] = df["EMRP_geo"].shift(-1)   # predict next month's excess return


# ============================================================
# STEP 2 — BUILD MACRO PREDICTORS (MONTH-END)
# ============================================================

def monthly_last(path, date_col="date", value_col="value", new_name=None):
    df = pd.read_csv(path, parse_dates=[date_col])
    df = df[[date_col, value_col]].set_index(date_col).sort_index()
    m = df.resample("M").last()
    return m.rename(columns={value_col: new_name})

# YoY series
cpi = monthly_last(paths["cpi"], value_col="pct_change_yoy", new_name="cpi_yoy")
pce = monthly_last(paths["pce"], value_col="pct_change_yoy", new_name="pce_yoy")
ppi = monthly_last(paths["ppi"], value_col="pct_change_yoy", new_name="ppi_yoy")
indprod = monthly_last(paths["indprod"], value_col="pct_change_yoy", new_name="indprod_yoy")
retail = monthly_last(paths["retail_sales"], value_col="pct_change_yoy", new_name="retail_sales_yoy")
invent = monthly_last(paths["inventories"], value_col="pct_change_yoy", new_name="inventories_yoy")
export_px = monthly_last(paths["export_px"], value_col="pct_change_yoy", new_name="export_px_yoy")
import_px = monthly_last(paths["import_px"], value_col="pct_change_yoy", new_name="import_px_yoy")
m2 = monthly_last(paths["m2"], value_col="pct_change_yoy", new_name="m2_yoy")

# Levels
unemp = monthly_last(paths["unemp"], value_col="value", new_name="unemp_rate")
fedfunds = monthly_last(paths["fedfunds"], value_col="value", new_name="fedfunds")
discount_rate = monthly_last(paths["discount_rate"], value_col="value", new_name="discount_rate")
spread = monthly_last(paths["spread_10y_2y"], value_col="value", new_name="spread_10y_2y")
nat_fin = monthly_last(paths["nat_fin_cond"], value_col="value", new_name="nat_fin_cond")
nasdaq_vol = monthly_last(paths["nasdaq_vol"], value_col="value", new_name="nasdaq_vol")
hy = monthly_last(paths["hy_spread"], value_col="value", new_name="hy_spread")
y2 = monthly_last(paths["y2"], value_col="value", new_name="y2")
y3m = monthly_last(paths["y3m"], value_col="value", new_name="y3m")
y10 = monthly_last(paths["y10"], value_col="value", new_name="y10")

macro = pd.concat([
    cpi, pce, ppi, indprod, retail, invent,
    export_px, import_px,
    unemp, m2, fedfunds, discount_rate,
    spread, nat_fin, nasdaq_vol, hy, y2, y3m, y10
], axis=1)

# Final merged dataset
reg_df = df.join(macro, how="inner").dropna()


# ============================================================
# STEP 3 — MULTIVARIATE OLS
# ============================================================

predictors = list(macro.columns)

df_multi = reg_df[["EMRP_next"] + predictors].dropna()

y = df_multi["EMRP_next"]
X = sm.add_constant(df_multi[predictors])

model_multi = sm.OLS(y, X).fit()

print("\n\n=== MULTIPLE OLS REGRESSION ===")
print(model_multi.summary())


# ============================================================
# STEP 4 — UNIVARIATE (SIMPLE) OLS FOR EACH VARIABLE
# ============================================================

simple_rows = []

for var in predictors:
    tmp = reg_df[["EMRP_next", var]].dropna()

    y_s = tmp["EMRP_next"]
    X_s = sm.add_constant(tmp[[var]])

    res = sm.OLS(y_s, X_s).fit()

    simple_rows.append({
        "variable": var,
        "coef": res.params[var],
        "t_stat": res.tvalues[var],
        "p_value": res.pvalues[var],
        "R_squared": res.rsquared,
        "n_obs": int(res.nobs),
        "sign": "positive" if res.params[var] > 0 else "negative"
    })

simple_df = pd.DataFrame(simple_rows).sort_values("p_value")
print("\n\n=== SIMPLE UNIVARIATE RESULTS ===")
print(simple_df.to_string(index=False))


# ============================================================
# STEP 5 — LASSO SELECTION + REDUCED OLS
# ============================================================

X_full = reg_df[predictors].values
y_full = reg_df["EMRP_next"].values

lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=5, random_state=0))
])

lasso_pipe.fit(X_full, y_full)

lasso = lasso_pipe.named_steps["lasso"]
lasso_alpha = lasso.alpha_
coefs = lasso.coef_

selected = [p for p, c in zip(predictors, coefs) if abs(c) > 1e-6]

print("\n\n=== LASSO RESULTS ===")
print("Chosen alpha:", lasso_alpha)
print("Selected variables:", selected)

# Reduced OLS
df_reduced = reg_df[["EMRP_next"] + selected].dropna()
y_r = df_reduced["EMRP_next"]
X_r = sm.add_constant(df_reduced[selected])

model_reduced = sm.OLS(y_r, X_r).fit()

print("\n\n=== REDUCED MULTIVARIATE OLS (POST-LASSO) ===")
print(model_reduced.summary())

/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  sp_m = sp.resample("M").last()[["sp500_geo"]]
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:67: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:83: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  m = df.resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:83: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  m = df.resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:83: FutureWarning



=== MULTIPLE OLS REGRESSION ===
                            OLS Regression Results                            
Dep. Variable:              EMRP_next   R-squared:                       0.181
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     2.797
Date:                Mon, 24 Nov 2025   Prob (F-statistic):           0.000208
Time:                        01:50:09   Log-Likelihood:                 452.41
No. Observations:                 247   AIC:                            -866.8
Df Residuals:                     228   BIC:                            -800.2
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const 